# FSD50K — mel-CNN training for CV Studio AudioClassification

Trains a mel-spectrogram CNN on [FSD50K](https://zenodo.org/record/4060432)
(200 AudioSet classes, ~51 k clips from Freesound).

Exports an ONNX model compatible with the CV Studio `AudioClassification` node:
- Input shape `(1, 1, 128, T_FIXED)` — `(batch, channels, n_mels, time)`
- Output shape `(1, 200)` — softmax class scores
- ONNX metadata key `names` → JSON class-name dict

> **FSD50K download** requires a free Zenodo account.
> Run `python download_fsd50k.py` (generated below) or download manually from
> https://zenodo.org/record/4060432 and set `DATASET_ROOT` accordingly.

> **Run on Google Colab** (free GPU): `Runtime → Change runtime type → T4 GPU`

In [ ]:
!pip install -q torch torchvision torchaudio librosa onnx onnxruntime pandas scikit-learn tqdm


In [ ]:
import os, json, random, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import onnx
import onnxruntime as ort
from tqdm.auto import tqdm

# ── Hyper-parameters (must match CV Studio AudioClassification node) ──
SR          = 22_050
N_MELS      = 128
N_FFT       = 2_048
HOP_LENGTH  = 512
MAX_SEC     = 5        # clips are padded/cropped to this duration
N_CLASSES   = 200

T_FRAMES    = 1 + int(SR * MAX_SEC) // HOP_LENGTH  # 216
print(f'Mel shape per clip: ({N_MELS}, {T_FRAMES})')

BATCH_SIZE  = 32
EPOCHS      = 50
LR          = 1e-3
ONNX_PATH   = 'fsd50k_cvstudio.onnx'

# Point this to the extracted FSD50K directory
DATASET_ROOT     = 'FSD50K'
TRAIN_AUDIO_DIR  = os.path.join(DATASET_ROOT, 'FSD50K.dev_audio')
EVAL_AUDIO_DIR   = os.path.join(DATASET_ROOT, 'FSD50K.eval_audio')
TRAIN_CSV        = os.path.join(DATASET_ROOT, 'FSD50K.ground_truth', 'dev.csv')
EVAL_CSV         = os.path.join(DATASET_ROOT, 'FSD50K.ground_truth', 'eval.csv')
VOCAB_CSV        = os.path.join(DATASET_ROOT, 'FSD50K.ground_truth', 'vocabulary.csv')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)


## Download instructions

FSD50K is hosted on Zenodo (DOI 10.5281/zenodo.4060432).
Download the following archives and extract them under a `FSD50K/` folder:

| File | Description |
|------|-------------|
| `FSD50K.dev_audio.zip` (parts 1-5) | Training audio (~25 GB) |
| `FSD50K.eval_audio.zip` | Evaluation audio (~10 GB) |
| `FSD50K.ground_truth.zip` | Metadata / labels |

```bash
# Using the Zenodo REST API:
# https://zenodo.org/record/4060432/files/<filename>?download=1
```

Set `DATASET_ROOT` in the cell above to the folder containing the extracted archives.


In [ ]:
# ── Build class-name mapping from vocabulary.csv ──
vocab_df  = pd.read_csv(VOCAB_CSV, names=['index', 'mid', 'name'])
# FSD50K vocabulary: index 0-199, AudioSet MID, display name
FSD50K_CLASS_NAMES = {int(row['index']): row['name'] for _, row in vocab_df.iterrows()}
print(f'{len(FSD50K_CLASS_NAMES)} classes loaded from vocabulary.')
print('Sample:', {k: FSD50K_CLASS_NAMES[k] for k in list(FSD50K_CLASS_NAMES)[:6]})


In [ ]:
def parse_fsd50k_csv(csv_path):
    """Parse FSD50K ground-truth CSV; return list of (filename, [label_ids])."""
    df = pd.read_csv(csv_path)
    # dev.csv columns: fname, labels (comma-separated MIDs), mids, split
    mid_to_idx = {v: k for k, v in
                  pd.read_csv(VOCAB_CSV, names=['index','mid','name'])
                  .set_index('mid')['index'].to_dict().items()}
    records = []
    for _, row in df.iterrows():
        fname   = str(int(row['fname'])) + '.wav'
        mids    = str(row['mids']).split(',')
        label_ids = [mid_to_idx[m.strip()] for m in mids if m.strip() in mid_to_idx]
        if label_ids:
            records.append((fname, label_ids))
    return records


class FSD50KDataset(Dataset):
    """Single-label subset of FSD50K (uses first label only for simplicity)."""

    def __init__(self, records, audio_dir, augment=False):
        self.records   = records
        self.audio_dir = audio_dir
        self.augment   = augment
        self.max_len   = SR * MAX_SEC

    def __len__(self):
        return len(self.records)

    def _load_mel(self, path):
        y, _ = librosa.load(path, sr=SR, mono=True)
        if len(y) < self.max_len:
            y = np.pad(y, (0, self.max_len - len(y)))
        else:
            y = y[:self.max_len]
        if self.augment and random.random() < 0.4:
            shift = random.randint(-SR // 2, SR // 2)
            y = np.roll(y, shift)
        mel    = librosa.feature.melspectrogram(
            y=y, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
        mel_db = librosa.power_to_db(mel).astype(np.float32)
        return mel_db

    def __getitem__(self, idx):
        fname, label_ids = self.records[idx]
        path  = os.path.join(self.audio_dir, fname)
        mel   = torch.tensor(self._load_mel(path)).unsqueeze(0)
        label = int(label_ids[0])  # primary label
        return mel, label


train_records = parse_fsd50k_csv(TRAIN_CSV)
eval_records  = parse_fsd50k_csv(EVAL_CSV)
print(f'Train: {len(train_records)} | Eval: {len(eval_records)}')

train_ds = FSD50KDataset(train_records, TRAIN_AUDIO_DIR, augment=True)
val_ds   = FSD50KDataset(eval_records,  EVAL_AUDIO_DIR,  augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
class MelCNN(nn.Module):
    def __init__(self, n_mels=N_MELS, n_classes=N_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(512, n_classes),
        )
    def forward(self, x): return self.classifier(self.features(x))


model = MelCNN().to(DEVICE)
print(f'Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss()

best_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = correct = total = 0
    for mels, labels in tqdm(train_loader, desc=f'Ep {epoch}', leave=False):
        mels, labels = mels.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits = model(mels)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * mels.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += mels.size(0)
    scheduler.step()

    model.eval()
    v_correct = v_total = 0
    with torch.no_grad():
        for mels, labels in val_loader:
            mels, labels = mels.to(DEVICE), labels.to(DEVICE)
            v_correct += (model(mels).argmax(1) == labels).sum().item()
            v_total   += mels.size(0)

    val_acc = v_correct / v_total
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'fsd50k_best.pth')

    if epoch % 5 == 0:
        print(f'Ep {epoch:3d} | loss {total_loss/total:.4f} | val {val_acc:.4f} | best {best_acc:.4f}')

print(f'Best val accuracy: {best_acc:.4f}')


In [ ]:
model.load_state_dict(torch.load('fsd50k_best.pth', map_location='cpu'))
model.eval()

class _WithSoftmax(nn.Module):
    def __init__(self, base): super().__init__(); self.base = base; self.sm = nn.Softmax(dim=1)
    def forward(self, x): return self.sm(self.base(x))

export_model = _WithSoftmax(model).eval()
dummy = torch.zeros(1, 1, N_MELS, T_FRAMES)

torch.onnx.export(
    export_model, dummy, ONNX_PATH,
    opset_version=13,
    input_names=['mel_input'],
    output_names=['class_scores'],
    dynamic_axes=None,
)

# Embed class names
proto = onnx.load(ONNX_PATH)
m = proto.metadata_props.add()
m.key   = 'names'
m.value = json.dumps({str(k): v for k, v in FSD50K_CLASS_NAMES.items()})
onnx.save(proto, ONNX_PATH)

print(f'Exported with {len(FSD50K_CLASS_NAMES)} class names → {ONNX_PATH}')

# Verify
sess      = ort.InferenceSession(ONNX_PATH, providers=['CPUExecutionProvider'])
meta      = sess.get_modelmeta().custom_metadata_map
print('Input :', sess.get_inputs()[0].shape)
print('Output:', sess.get_outputs()[0].shape)
print(f'Classes embedded: {len(json.loads(meta["names"]))}')
print(f'\n✓ Upload "{ONNX_PATH}" via the 📂 Add Model button in CV Studio!')
